In [10]:
import logging, sys, warnings
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

In [11]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "bigdata" / "processed"
REPORTS_DIR = BASE_DIR / "artifacts" / "eda"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = REPORTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [12]:
def save_fig(name, show=True):
    plt.tight_layout()
    if show:
        plt.show()
    plt.savefig(FIGURES_DIR / name, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f"Сохранён график: {name}")

In [13]:
df = pd.read_parquet(PROCESSED_DIR / "combined_features.parquet")

feature_cols = []
if (PROCESSED_DIR / "feature_columns.txt").exists():
    with open(PROCESSED_DIR / "feature_columns.txt") as f:
        feature_cols = [line.strip() for line in f if line.strip()]
    feature_cols = [c for c in feature_cols if c in df.columns]

logger.info(f"Датасет: {df.shape}, тикеров: {df['ticker'].nunique()}, фичей: {len(feature_cols)}")

2026-05-24 16:54:59,792 [INFO] Датасет: (766539, 124), тикеров: 249, фичей: 109


In [14]:
df

,date,open,high,low,close,volume,ticker,sma_20,sma_50,sma_200,...,day_of_month,week_of_year,days_to_month_end,days_from_month_start,day_of_week_sin,day_of_week_cos,month_sin,month_cos,is_quarter_end,is_month_end
0,1999-06-01,228.500,228.500,223.100,225.000,12471,LKOH,NaN,NaN,NaN,...,1,22,29,1,0.781831,0.62349,1.224647e-16,-1.0,1,0
1,1999-06-01,0.675,0.690,0.659,0.660,2007,MSNG,NaN,NaN,NaN,...,1,22,29,1,0.781831,0.62349,1.224647e-16,-1.0,1,0
2,1999-06-01,28.220,28.220,28.000,28.000,2,RTKM,NaN,NaN,NaN,...,1,22,29,1,0.781831,0.62349,1.224647e-16,-1.0,1,0
3,1999-06-01,0.540,0.540,0.530,0.530,778,SBER,NaN,NaN,NaN,...,1,22,29,1,0.781831,0.62349,1.224647e-16,-1.0,1,0
4,1999-06-01,0.290,0.290,0.280,0.280,40200,SBERP,NaN,NaN,NaN,...,1,22,29,1,0.781831,0.62349,1.224647e-16,-1.0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766534,2024-08-27,0.442,0.448,0.435,0.438,213,YKENP,0.4588,0.46594,0.580385,...,27,35,4,27,0.781831,0.62349,-8.660254e-01,-0.5,0,0
766535,2024-08-27,882.000,882.000,858.000,862.000,13,YRSB,888.2000,888.88000,1097.380000,...,27,35,4,27,0.781831,0.62349,-8.660254e-01,-0.5,0,0
766536,2024-08-27,219.000,220.500,217.000,220.500,60,YRSBP,223.0750,226.34000,266.877500,...,27,35,4,27,0.781831,0.62349,-8.660254e-01,-0.5,0,0
766537,2024-08-27,3150.000,3230.000,3115.000,3115.000,198,ZILL,3225.5000,3284.20000,3327.375000,...,27,35,4,27,0.781831,0.62349,-8.660254e-01,-0.5,0,0


In [15]:
"""Анализ качества данных: пропуски, дубликаты, выбросы."""
logger.info("\n=== 1. КАЧЕСТВО ДАННЫХ ===")
report = []

# Пропуски
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df = missing_df[missing_df['missing_count'] > 0].sort_values('missing_pct', ascending=False)

logger.info(f"Колонок с пропусками: {len(missing_df)}")
if len(missing_df) > 0:
    logger.info(f"Топ-10 пропусков:\n{missing_df.head(10)}")
    missing_df.to_csv(REPORTS_DIR / "missing_values.csv")

# Дубликаты
dups = df.duplicated(subset=['ticker', 'date']).sum()
logger.info(f"Дубликатов (ticker, date): {dups}")

# Нулевые объёмы
zero_vol = (df['volume'] == 0).sum()
logger.info(f"Нулевых объёмов: {zero_vol}")

missing_df

2026-05-24 16:54:59,957 [INFO] 
=== 1. КАЧЕСТВО ДАННЫХ ===
2026-05-24 16:55:00,100 [INFO] Колонок с пропусками: 74
2026-05-24 16:55:00,105 [INFO] Топ-10 пропусков:
                    missing_count  missing_pct
target_return_200d          49205         6.42
sma_200                     48962         6.39
price_sma200_dev            48962         6.39
vol_ratio_50                15513         2.02
price_sma50_dev             12097         1.58
vol_ma50                    12097         1.58
sma_50                      12097         1.58
volume_change_1d             9398         1.23
vol_ratio_20                 8273         1.08
volatility_20d               4963         0.65
2026-05-24 16:55:00,156 [INFO] Дубликатов (ticker, date): 0
2026-05-24 16:55:00,156 [INFO] Нулевых объёмов: 9149


,missing_count,missing_pct
target_return_200d,49205,6.42
sma_200,48962,6.39
price_sma200_dev,48962,6.39
vol_ratio_50,15513,2.02
price_sma50_dev,12097,1.58
...,...,...
usd_rub_vol_20d_delta3m,43,0.01
usd_rub_dev_from_ma20_delta3m,43,0.01
log_ret_lag_5d,4,0.00
market_log_ret_1d,7,0.00


Пропуски присутствуют в 74 колонках, но они ожидаемы: target_return_200d и sma_200 имеют пропуски на последних и первых 200 днях соответственно, аналогично для индикаторов с периодами 50, 20 и 14 дней. Макро‑данные и новостной сентимент пропусков не содержат. Дубликатов нет. Нулевые объёмы (1,19% дней) могут означать дни без торгов (возможно для неликвидных акций).

In [ ]:
# Анализ выбросов во всех числовых признаках
logger.info("\n=== ВЫБРОСЫ ВО ВСЕХ ПРИЗНАКАХ (метод 3*IQR) ===")

outlier_stats = []
for col in feature_cols:
    if df[col].dtype in ['float64', 'int64']:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        if IQR == 0:
            continue
        outliers = ((df[col] < Q1 - 3*IQR) | (df[col] > Q3 + 3*IQR)).sum()
        outlier_pct = outliers / len(df) * 100
        outlier_stats.append((col, outliers, outlier_pct))

outlier_df = pd.DataFrame(outlier_stats, columns=['feature', 'outliers', 'pct'])
outlier_df = outlier_df.sort_values('pct', ascending=False)

logger.info(f"Признаков с выбросами: {len(outlier_df)}")
if not outlier_df.empty:
    logger.info(f"Топ-10 признаков по доле выбросов:\n{outlier_df.head(10)}")
    outlier_df.to_csv(REPORTS_DIR / "outliers_all_features.csv", index=False)

# Boxplot для топ-5 признаков с наибольшим количеством выбросов
if not outlier_df.empty:
    top_outlier_features = outlier_df.head(5)['feature'].tolist()
    fig, axes = plt.subplots(1, len(top_outlier_features), figsize=(5*len(top_outlier_features), 6))
    if len(top_outlier_features) == 1:
        axes = [axes]
    for ax, feat in zip(axes, top_outlier_features):
        sns.boxplot(y=df[feat].dropna().sample(min(10000, len(df))), ax=ax, color='salmon')
        ax.set_title(feat, fontsize=10)
        ax.set_ylabel('')
    plt.suptitle('Boxplot признаков с наибольшим количеством выбросов (3*IQR)')
    save_fig("11_outliers_boxplots.png")

2026-05-24 16:55:00,177 [INFO] 
=== ВЫБРОСЫ ВО ВСЕХ ПРИЗНАКАХ (метод 3*IQR) ===
2026-05-24 16:55:02,048 [INFO] Признаков с выбросами: 100
2026-05-24 16:55:02,048 [INFO] Топ-10 признаков по доле выбросов:
                   feature  outliers        pct
8                macd_hist    219512  28.636769
6                     macd    219043  28.575585
7              macd_signal    218436  28.496397
62  macro_key_rate_delta3m    179605  23.430641
92              news_count    139791  18.236645
28                     obv    135195  17.637067
43           volume_lag_1d    130487  17.022878
17                vol_ma20    126831  16.545929
18                vol_ma50    124723  16.270927
12                     atr     88004  11.480694
2026-05-24 16:55:02,704 [INFO] Сохранён график: 11_outliers_boxplots.png


Выбросы по ценам, определённые через 3×IQR, составляют до 28,4% — это много, но объясняется разным масштабом цен (от копеек до десятков тысяч рублей) и реальными рыночными движениями. Удалять такие выбросы не нужно. 

In [17]:
"""Распределения ключевых переменных."""
logger.info("\n=== 2. РАСПРЕДЕЛЕНИЯ ===")

# Цены (лог-масштаб)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, col in enumerate(['open', 'high', 'low', 'close']):
    ax = axes[idx // 2, idx % 2]
    sample = df[col].dropna().sample(min(50000, len(df)))
    sns.histplot(sample, bins=100, kde=True, ax=ax)
    ax.set_title(f'Распределение {col}')
    ax.set_xlabel(col)
save_fig("01_price_distributions.png")

# Лог-доходности
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
log_ret = df['log_ret'].dropna()
sns.histplot(log_ret, bins=200, kde=True, ax=axes[0])
axes[0].set_title('Распределение лог-доходности')
axes[0].set_xlim(-0.2, 0.2)

# QQ-plot для нормальности
stats.probplot(log_ret.sample(10000), dist="norm", plot=axes[1])
axes[1].set_title('QQ-plot лог-доходности')
save_fig("02_log_returns_distribution.png")

# Статистики
logger.info(f"Лог-доходность: mean={log_ret.mean():.6f}, std={log_ret.std():.6f}")
logger.info(f"Скошенность: {stats.skew(log_ret):.4f}, Эксцесс: {stats.kurtosis(log_ret):.4f}")

# Объём
fig, ax = plt.subplots(figsize=(10, 5))
vol_sample = df['volume'].dropna().sample(min(50000, len(df)))
sns.histplot(vol_sample, bins=100, kde=True, ax=ax)
ax.set_title('Распределение объёма')
ax.set_xlabel('Volume')
save_fig("03_volume_distribution.png")

2026-05-24 16:55:02,708 [INFO] 
=== 2. РАСПРЕДЕЛЕНИЯ ===
2026-05-24 16:55:04,226 [INFO] Сохранён график: 01_price_distributions.png
2026-05-24 16:55:05,690 [INFO] Сохранён график: 02_log_returns_distribution.png
2026-05-24 16:55:05,706 [INFO] Лог-доходность: mean=0.000275, std=0.056996
2026-05-24 16:55:05,740 [INFO] Скошенность: 0.2975, Эксцесс: 3217.2172
2026-05-24 16:55:06,088 [INFO] Сохранён график: 03_volume_distribution.png


Цены распределены экспоненциально: большинство акций дешёвые, дорогих мало. Лог‑доходность в среднем равна 0,000275 (около 0,0275% в день), стандартное отклонение — 0,057 (5,7% в день), что говорит о высокой волатильности. Скошенность равна 0,30 (слабый правый хвост), но эксцесс достигает 3217 — это аномалия, вызванная экстремальными выбросами, возможно, из‑за корпоративных действий или ошибок данных. Обрезать выбросы не планируется: модель должна видеть кризисные сценарии.

In [18]:
"""Сезонность и временные паттерны."""
logger.info("\n=== 3. СЕЗОННОСТЬ ===")

# Доходность по дням недели
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

dow_returns = df.groupby('day_of_week')['log_ret'].mean()
dow_returns.plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Средняя доходность по дням недели')
axes[0,0].set_xlabel('День недели (0=Пн)')
axes[0,0].tick_params(axis='x', rotation=0)

# По месяцам
month_returns = df.groupby('month')['log_ret'].mean()
month_returns.plot(kind='bar', ax=axes[0,1], color='coral')
axes[0,1].set_title('Средняя доходность по месяцам')
axes[0,1].set_xlabel('Месяц')

# По кварталам
quarter_returns = df.groupby('quarter')['log_ret'].mean()
quarter_returns.plot(kind='bar', ax=axes[1,0], color='green')
axes[1,0].set_title('Средняя доходность по кварталам')
axes[1,0].set_xlabel('Квартал')

# Волатильность по дням недели
dow_vol = df.groupby('day_of_week')['volatility_20d'].mean()
dow_vol.plot(kind='bar', ax=axes[1,1], color='purple')
axes[1,1].set_title('Средняя волатильность по дням недели')
axes[1,1].set_xlabel('День недели')

save_fig("04_seasonality.png")

logger.info(f"Доходность по дням недели:\n{dow_returns}")
logger.info(f"Доходность по месяцам:\n{month_returns}")

2026-05-24 16:55:06,103 [INFO] 
=== 3. СЕЗОННОСТЬ ===
2026-05-24 16:55:06,606 [INFO] Сохранён график: 04_seasonality.png
2026-05-24 16:55:06,606 [INFO] Доходность по дням недели:
day_of_week
0    0.000718
1    0.000274
2    0.000722
3   -0.000811
4    0.000404
5    0.003677
6    0.011405
Name: log_ret, dtype: float64
2026-05-24 16:55:06,606 [INFO] Доходность по месяцам:
month
1     0.002523
2    -0.000321
3     0.000757
4     0.000490
5    -0.001044
6    -0.000060
7     0.000528
8     0.001412
9    -0.000578
10   -0.000226
11   -0.000139
12   -0.000042
Name: log_ret, dtype: float64


В выходные дни (суббота и воскресенье) в данных наблюдается положительная доходность — это не ошибка, а отражение гэпа открытия в понедельник («эффект выходных»). Удалять выходные не следует, признак day_of_week уже содержит эту информацию. По месяцам подтверждается «январский эффект» (январь — лучший месяц) и слабость мая («Sell in May»). Эти паттерны уже закодированы через тригонометрические признаки month_sin/cos. Волатильность, рассчитанная как скользящее стандартное отклонение, повышается в воскресенье из‑за включения пятничных данных и ожидаемого гэпа понедельника — это полезная предсказательная информация.

In [19]:
"""Корреляционный анализ."""
logger.info("\n=== 4. КОРРЕЛЯЦИИ ===")

# Корреляция с таргетами
target_cols = [c for c in df.columns if c.startswith('target_')]
corr_with_target = df[feature_cols + target_cols].corr()

# Топ коррелирующие с target_return_20d
if 'target_return_20d' in corr_with_target.columns:
    top_corr = corr_with_target['target_return_20d'].drop(target_cols).abs().sort_values(ascending=False).head(20)
    logger.info(f"Топ-20 корреляций с target_return_20d:\n{top_corr}")
    top_corr.to_csv(REPORTS_DIR / "top_correlations_return20.csv")

    fig, ax = plt.subplots(figsize=(10, 8))
    top_corr.plot(kind='barh', ax=ax, color='teal')
    ax.set_title('Top-20 |корреляций| с target_return_20d')
    save_fig("05_top_correlations.png")

# Матрица корреляций макро
macro_cols = [c for c in feature_cols if 'macro' in c or 'usd_rub' in c or 'm2' in c]
if len(macro_cols) > 1:
    fig, ax = plt.subplots(figsize=(10, 8))
    macro_corr = df[macro_cols].corr()
    sns.heatmap(macro_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
    ax.set_title('Корреляция макро-фичей')
    save_fig("06_macro_correlation_matrix.png")

2026-05-24 16:55:06,627 [INFO] 
=== 4. КОРРЕЛЯЦИИ ===
2026-05-24 16:55:25,451 [INFO] Топ-20 корреляций с target_return_20d:
body_size                        0.154190
intraday_range                   0.151915
relative_log_ret_1d              0.093207
usd_rub_delta1m                  0.092235
log_ret                          0.080266
usd_rub_delta3m                  0.079096
usd_rub_dev_from_ma20            0.077191
relative_return_20d              0.076520
lower_shadow                     0.058336
sma_50_200_cross                 0.046545
usd_rub_vol_20d                  0.046069
m2_growth_3m_lag1m               0.044671
sma_20_50_cross                  0.043940
rsi                              0.043146
usd_rub                          0.040691
cci                              0.040168
usd_rub_dev_from_ma20_lag1m      0.039956
price_above_sma20                0.039830
macro_inflation_delta3m          0.039565
usd_rub_dev_from_ma20_delta3m    0.039059
Name: target_return_20d, dtype: floa

Самые высокие абсолютные корреляции с целевой доходностью за 20 дней показывают внутридневные паттерны: body_size (размер тела свечи) и intraday_range (диапазон дня) — оба около 0,15. Затем следуют относительная доходность, изменения курса USD/RUB за месяц и три месяца, отклонения курса от скользящей средней, а также технические индикаторы вроде пересечения SMA 50/200 и RSI. Макро‑признаки в топ‑20 представлены именно изменениями и отклонениями, а не уровнями (например, usd_rub_delta1m имеет корреляцию 0,092 против 0,041 у абсолютного курса). Это подтверждает, что для модели важны дельты и аномалии макро‑переменных. Новостной сентимент и объём не попали в топ‑20, что нормально: они могут работать через нелинейные взаимодействия.

In [20]:
"""Влияние макро-переменных на рынок."""
logger.info("\n=== 5. МАКРО ВЛИЯНИЕ ===")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# USD/RUB vs доходность
ax = axes[0,0]
sample = df.sample(min(20000, len(df)))
ax.scatter(sample['usd_rub'], sample['log_ret'], alpha=0.1, s=1)
ax.set_xlabel('USD/RUB')
ax.set_ylabel('Лог-доходность')
ax.set_title('USD/RUB vs Лог-доходность')
ax.set_ylim(-0.1, 0.1)

# Ключевая ставка vs доходность
ax = axes[0,1]
ax.scatter(sample['macro_key_rate'], sample['log_ret'], alpha=0.1, s=1, color='red')
ax.set_xlabel('Ключевая ставка')
ax.set_ylabel('Лог-доходность')
ax.set_title('Key Rate vs Лог-доходность')
ax.set_ylim(-0.1, 0.1)

# Инфляция vs доходность
ax = axes[1,0]
ax.scatter(sample['macro_inflation'], sample['log_ret'], alpha=0.1, s=1, color='green')
ax.set_xlabel('Инфляция % г/г')
ax.set_ylabel('Лог-доходность')
ax.set_title('Inflation vs Лог-доходность')
ax.set_ylim(-0.1, 0.1)

# M2 vs доходность
ax = axes[1,1]
ax.scatter(sample['macro_money_supply'], sample['log_ret'], alpha=0.1, s=1, color='purple')
ax.set_xlabel('M2 (млрд руб)')
ax.set_ylabel('Лог-доходность')
ax.set_title('M2 vs Лог-доходность')
ax.set_ylim(-0.1, 0.1)

save_fig("07_macro_impact.png")

# Корреляции
macro_cols = ['usd_rub', 'macro_key_rate', 'macro_inflation', 'macro_money_supply']
for col in macro_cols:
    if col in df.columns:
        corr = df[col].corr(df['log_ret'])
        logger.info(f"Корреляция {col} с log_ret: {corr:.4f}")

2026-05-24 16:55:29,245 [INFO] 
=== 5. МАКРО ВЛИЯНИЕ ===
2026-05-24 16:55:30,093 [INFO] Сохранён график: 07_macro_impact.png
2026-05-24 16:55:30,122 [INFO] Корреляция usd_rub с log_ret: 0.0072
2026-05-24 16:55:30,125 [INFO] Корреляция macro_key_rate с log_ret: -0.0007
2026-05-24 16:55:30,141 [INFO] Корреляция macro_inflation с log_ret: -0.0019
2026-05-24 16:55:30,157 [INFO] Корреляция macro_money_supply с log_ret: 0.0028


Линейные корреляции макро‑уровней с дневной доходностью близки к нулю: USD/RUB даёт 0,0072, ключевая ставка −0,0007, инфляция −0,0019, денежная масса 0,0028. Это не означает бесполезность макро — эффект нелинейный и лагированный. На графиках scatter видны кластеры и структурные сдвиги (например, дискретные изменения ставки). Сильные корреляции дают именно производные признаки (дельта за месяц, отклонение от скользящей средней, рост денежной массы с лагом).

In [21]:
"""Анализ новостного сентимента."""
logger.info("\n=== 6. НОВОСТНОЙ СЕНТИМЕНТ ===")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Распределение сентимента
sentiment = df['news_sentiment'].dropna()
sns.histplot(sentiment, bins=50, kde=True, ax=axes[0])
axes[0].set_title('Распределение новостного сентимента')
axes[0].axvline(sentiment.mean(), color='r', linestyle='--', label=f'mean={sentiment.mean():.3f}')
axes[0].legend()

# Сентимент vs доходность
sample = df.dropna(subset=['news_sentiment', 'log_ret']).sample(min(20000, len(df)))
axes[1].scatter(sample['news_sentiment'], sample['log_ret'], alpha=0.1, s=1)
axes[1].set_xlabel('News Sentiment')
axes[1].set_ylabel('Лог-доходность')
axes[1].set_title('Sentiment vs Лог-доходность')
axes[1].set_ylim(-0.1, 0.1)

save_fig("08_news_sentiment.png")

corr = df['news_sentiment'].corr(df['log_ret'])
logger.info(f"Корреляция sentiment с log_ret: {corr:.4f}")
logger.info(f"Средний sentiment: {sentiment.mean():.4f}, std: {sentiment.std():.4f}")

2026-05-24 16:55:30,182 [INFO] 
=== 6. НОВОСТНОЙ СЕНТИМЕНТ ===
2026-05-24 16:55:36,448 [INFO] Сохранён график: 08_news_sentiment.png
2026-05-24 16:55:36,465 [INFO] Корреляция sentiment с log_ret: 0.0172
2026-05-24 16:55:36,475 [INFO] Средний sentiment: -0.0199, std: 0.5757


Распределение сентимента мультимодальное с пиками около −0,75 (сильно негативные новости), 0 (нейтральные), +0,5 и +0,9 (позитивные). Среднее значение слегка отрицательное (−0,020). Корреляция с дневной доходностью составляет всего 0,017 — очень слабая. Причина в том, что новости агрегированы по всем тикерам и не специфичны для отдельной компании. Тем не менее сентимент оставлен в данных, так как даже слабый сигнал полезен в ансамбле с другими признаками, а нелинейные пороговые эффекты могут быть пойманы моделями.

In [22]:
"""Анализ таргетов: баланс классов, распределение доходностей."""
logger.info("\n=== 7. АНАЛИЗ ТАРГЕТОВ ===")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, h in enumerate([1, 5, 20, 200]):
    ax = axes[idx // 2, idx % 2]
    col = f'target_return_{h}d'
    if col in df.columns:
        data = df[col].dropna()
        sns.histplot(data, bins=200, kde=True, ax=ax)
        ax.set_title(f'Распределение target_return_{h}d')
        ax.set_xlim(-0.5, 0.5)
        logger.info(f"{col}: mean={data.mean():.6f}, std={data.std():.4f}, "
                    f"skew={stats.skew(data):.4f}")

save_fig("09_target_distributions.png")

# Баланс классов
for h in [1, 5, 20, 200]:
    col = f'target_binary_{h}d'
    if col in df.columns:
        counts = df[col].value_counts(normalize=True)
        logger.info(f"{col}: {counts.to_dict()}")

2026-05-24 16:55:36,485 [INFO] 
=== 7. АНАЛИЗ ТАРГЕТОВ ===
2026-05-24 16:55:37,633 [INFO] target_return_1d: mean=0.000275, std=0.0570, skew=0.2975
2026-05-24 16:55:38,803 [INFO] target_return_5d: mean=0.001382, std=0.0975, skew=1.3214
2026-05-24 16:55:40,165 [INFO] target_return_20d: mean=0.005947, std=0.1754, skew=0.7299
2026-05-24 16:55:42,210 [INFO] target_return_200d: mean=0.075885, std=0.5251, skew=0.2695
2026-05-24 16:55:42,953 [INFO] Сохранён график: 09_target_distributions.png
2026-05-24 16:55:42,969 [INFO] target_binary_1d: {0: 0.5453721206618319, 1: 0.45462787933816806}
2026-05-24 16:55:42,969 [INFO] target_binary_5d: {0: 0.5244534198520884, 1: 0.47554658014791157}
2026-05-24 16:55:42,969 [INFO] target_binary_20d: {0: 0.5075971346532923, 1: 0.49240286534670774}
2026-05-24 16:55:42,984 [INFO] target_binary_200d: {1: 0.5225996328953908, 0: 0.47740036710460915}


Средняя доходность растёт с горизонтом: 0,0275% за 1 день, 0,138% за 5 дней, 0,595% за 20 дней, 7,59% за 200 дней. Стандартное отклонение также увеличивается (5,7%, 9,8%, 17,5%, 52,5%) — в соответствии с теорией. Скошенность максимальна на 5‑дневном горизонте (1,32), что говорит о более частых сильных положительных движениях. Баланс бинарных классов близок к 50/50: для 1 дня 54,5% движений вниз и 45,5% вверх, для 20 дней — 50,8% и 49,2%, для 200 дней — 47,7% и 52,3%.

In [23]:
"""Анализ тикеров: количество, длина истории, ликвидность."""
logger.info("\n=== 8. УНИВЕРСУМ ТИКЕРОВ ===")

ticker_stats = df.groupby('ticker').agg(
    n_days=('date', 'count'),
    first_date=('date', 'min'),
    last_date=('date', 'max'),
    avg_volume=('volume', 'mean'),
    avg_close=('close', 'mean')
).sort_values('n_days', ascending=False)

logger.info(f"Всего тикеров: {len(ticker_stats)}")
logger.info(f"Средняя длина истории: {ticker_stats['n_days'].mean():.0f} дней")
logger.info(f"Медианная длина: {ticker_stats['n_days'].median():.0f} дней")
logger.info(f"Топ-5 по длине истории:\n{ticker_stats.head()}")

ticker_stats.to_csv(REPORTS_DIR / "ticker_statistics.csv")

# Распределение длины истории
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(ticker_stats['n_days'], bins=50, kde=True, ax=ax)
ax.set_title('Распределение длины истории по тикерам')
ax.set_xlabel('Количество дней')
save_fig("10_ticker_history_length.png")

2026-05-24 16:55:42,994 [INFO] 
=== 8. УНИВЕРСУМ ТИКЕРОВ ===
2026-05-24 16:55:43,043 [INFO] Всего тикеров: 249
2026-05-24 16:55:43,043 [INFO] Средняя длина истории: 3078 дней
2026-05-24 16:55:43,043 [INFO] Медианная длина: 3221 дней
2026-05-24 16:55:43,050 [INFO] Топ-5 по длине истории:
        n_days first_date  last_date    avg_volume    avg_close
ticker                                                         
LKOH      6305 1999-06-01 2024-08-27  1.414817e+06  2605.778668
SNGS      6305 1999-06-01 2024-08-27  3.384118e+05    25.368590
SNGSP     6304 1999-06-01 2024-08-27  2.599754e+05    23.530984
RTKM      6295 1999-06-01 2024-08-27  3.429211e+05   100.991865
MSNG      6295 1999-06-01 2024-08-27  3.724884e+04     2.294369
2026-05-24 16:55:43,243 [INFO] Сохранён график: 10_ticker_history_length.png


Всего 249 тикеров. Средняя длина истории — 3078 дней (12,3 года), медианная — 3221 день. Есть тикеры с очень короткой историей (менее 500 дней) и с длинной (до 6300 дней). Обнаружены аномалии: отрицательный объём у TGKA (будет отфильтрован в prepare_data_v2), огромные средние объёмы у GMKN и TRNFP (требуют проверки исходных CSV), а также очень низкая цена TGKB (0,006 руб.) — она реальна, так как акции дроблёные. Для горизонта 200 дней стоит исключить тикеры с историей менее 1000 дней или использовать только общие признаки.